# Données météo et modèle VARIMA

Ce notebook ajoute la **température moyenne quotidienne des Hauts-de-France** à l'étude de la consommation électrique :

1. Chargement et fusion de la consommation journalière et de la température.
2. Lien entre température et consommation (thermosensibilité).
3. XGBoost avec et sans la température.
4. Modèle **VARIMA** sur le couple (consommation, température), comparé à un ARIMA sur la consommation seule.

**Données nécessaires** (dans le dossier `data/`, non versionné) :
- `data.csv` (consommation RTE) ou `hdf_daily.csv` (généré par le notebook `3_model`) ;
- `temperature-quotidienne-regionale.csv` : température quotidienne régionale (ODRE / data.gouv.fr), séparateur `;`, disponible depuis 2016.

## 1. Chargement des données

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams.update({'figure.figsize': (12, 5)})

DATA = Path('../data')
CODE_HAUTS_DE_FRANCE = 32
N_TEST = 730  # deux dernières années en test, comme dans 3_model


def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
# Consommation journalière : fichier produit par 3_model, sinon calculée depuis le CSV RTE
if (DATA / 'hdf_daily.csv').exists():
    conso = pd.read_csv(DATA / 'hdf_daily.csv', index_col='date_heure', parse_dates=True)
    conso = conso['consommation_brute_electricite_rte']
    conso.index = pd.to_datetime(conso.index, utc=True)
else:
    df = pd.read_csv(DATA / 'data.csv', sep=';', usecols=['date_heure', 'code_insee_region', 'consommation_brute_electricite_rte'])
    df = df[df['code_insee_region'] == CODE_HAUTS_DE_FRANCE]
    conso = df.set_index(pd.to_datetime(df['date_heure'], utc=True))['consommation_brute_electricite_rte'].sort_index()
    conso = conso.resample('D').mean()

# Index journalier sans fuseau horaire, pour le fusionner avec les dates de la météo
conso.index = conso.index.tz_localize(None).normalize()
conso.name = 'consommation'
conso

In [ ]:
# Température moyenne quotidienne, uniquement pour les Hauts-de-France
temp = pd.read_csv(DATA / 'temperature-quotidienne-regionale.csv', sep=';')
temp = temp[temp['Code INSEE région'] == CODE_HAUTS_DE_FRANCE]
temp = temp.set_index(pd.to_datetime(temp['Date']))['TMoy (°C)'].sort_index()
temp = temp[~temp.index.duplicated()]
temp.name = 'temperature'
temp

In [ ]:
# Fusion sur les dates communes, fréquence journalière et jours manquants interpolés
df_meteo = pd.concat([conso, temp], axis=1, join='inner').asfreq('D').interpolate()
print(f"Période commune : du {df_meteo.index[0]:%d/%m/%Y} au {df_meteo.index[-1]:%d/%m/%Y} ({len(df_meteo)} jours)")
df_meteo.describe()

## 2. Lien entre température et consommation

In [ ]:
# Deux graphiques superposés plutôt qu'un double axe : chaque série garde sa propre échelle
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
df_meteo['consommation'].plot(ax=axes[0], color='#2a78d6', lw=1)
axes[0].set_ylabel('Consommation (MW)')
axes[0].set_title('Consommation électrique journalière')
df_meteo['temperature'].plot(ax=axes[1], color='#eb6834', lw=1)
axes[1].set_ylabel('Température moyenne (°C)')
axes[1].set_title('Température moyenne quotidienne')
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
correlation = df_meteo['consommation'].corr(df_meteo['temperature'])
print(f"Corrélation linéaire entre température et consommation : {correlation:.2f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_meteo['temperature'], df_meteo['consommation'], s=8, alpha=0.4, color='#2a78d6')
ax.set_xlabel('Température moyenne (°C)')
ax.set_ylabel('Consommation (MW)')
ax.set_title('Consommation en fonction de la température')
ax.grid(True, alpha=0.3)
plt.show()

La consommation baisse quand la température monte : c'est l'effet du **chauffage électrique**. Le nuage de points montre en général une relation plus forte par temps froid, qui s'aplatit au-delà d'environ 15 °C (pas de besoin de chauffage).

## 3. XGBoost avec et sans la température

Même découpage que dans `3_model` : les deux dernières années servent de test.

La température utilisée sur la période de test est la **température observée**. En conditions réelles, il faudrait utiliser des prévisions météo : l'écart mesuré ici est donc un maximum de ce que la météo peut apporter.

In [ ]:
from xgboost import XGBRegressor

variables = pd.DataFrame({
    'dayofweek': df_meteo.index.dayofweek,
    'dayofyear': df_meteo.index.dayofyear,
    'month': df_meteo.index.month,
    'year': df_meteo.index.year,
    'temperature': df_meteo['temperature'],
}, index=df_meteo.index)
y = df_meteo['consommation']
y_train, y_test = y.iloc[:-N_TEST], y.iloc[-N_TEST:]

resultats = {}
previsions = {}
for nom, colonnes in {
    'XGBoost (calendrier)': ['dayofweek', 'dayofyear', 'month', 'year'],
    'XGBoost (calendrier + température)': ['dayofweek', 'dayofyear', 'month', 'year', 'temperature'],
}.items():
    modele = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=0)
    modele.fit(variables[colonnes].iloc[:-N_TEST], y_train)
    prevision = pd.Series(modele.predict(variables[colonnes].iloc[-N_TEST:]), index=y_test.index)
    previsions[nom] = prevision
    resultats[nom] = {
        'MAPE (%)': mean_absolute_percentage_error(y_test, prevision),
        'MAE (MW)': mean_absolute_error(y_test, prevision),
        'RMSE (MW)': np.sqrt(mean_squared_error(y_test, prevision)),
    }

pd.DataFrame(resultats).T.round(2)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test.index, y_test, label='Consommation réelle', color='#7a7974', lw=1)
for (nom, prevision), couleur in zip(previsions.items(), ['#2a78d6', '#eb6834']):
    plt.plot(prevision.index, prevision, label=f"{nom} (MAPE : {resultats[nom]['MAPE (%)']:.2f} %)", color=couleur, lw=1.5)
plt.legend()
plt.ylabel('Consommation (MW)')
plt.title('XGBoost : apport de la température')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Modèle VARIMA

Un **VAR** (Vector AutoRegression) modélise plusieurs séries ensemble : chaque série est expliquée par ses propres retards **et** par ceux des autres séries. Le **VARIMA** ajoute la différenciation (I) et une partie moyenne mobile (MA).

Ici, on modélise ensemble la consommation et la température. Comme pour ARIMA et SARIMA dans `3_model`, on travaille sur les séries **désaisonnalisées** : la saisonnalité annuelle de chaque série est estimée sur la période d'entraînement seule, puis réintégrée dans les prévisions de consommation.

In [ ]:
train, test = df_meteo.iloc[:-N_TEST], df_meteo.iloc[-N_TEST:]

# Saisonnalité annuelle de chaque série, estimée sur l'entraînement seul
saisonnalites = {col: seasonal_decompose(train[col], model='additive', period=365).seasonal for col in df_meteo.columns}
train_desaison = pd.DataFrame({col: train[col] - saisonnalites[col] for col in df_meteo.columns})


def saisonnalite_future(col, horizon):
    # seasonal_decompose répète le même motif tous les 365 points : on le prolonge sur la période de test
    n = len(train)
    return saisonnalites[col].iloc[[(n + h) % 365 for h in range(horizon)]].to_numpy()


train_desaison.plot(subplots=True, figsize=(12, 6), title='Séries désaisonnalisées (entraînement)')
plt.tight_layout()
plt.show()

### 4.1 Choix des ordres

In [ ]:
# d : on différencie une fois si l'une des séries n'est pas stationnaire (test de Dickey-Fuller)
p_valeurs = {col: adfuller(train_desaison[col])[1] for col in train_desaison.columns}
print('P-valeurs ADF :', {col: round(p, 4) for col, p in p_valeurs.items()})
d = 1 if max(p_valeurs.values()) > 0.05 else 0

# p : nombre de retards qui minimise l'AIC d'un VAR sur la série (différenciée si d = 1)
serie_var = train_desaison.diff().dropna() if d == 1 else train_desaison
selection = sm.tsa.VAR(serie_var).select_order(maxlags=14)
p = max(1, selection.aic)
print(selection.summary())
print(f"Ordres retenus : p = {p}, d = {d}, q = 0")

On garde **q = 0** : l'estimation d'un VARMA avec une partie moyenne mobile est lente et peu robuste (problèmes d'identification signalés par statsmodels). Un VARIMA(p, d, 0) est un VAR sur la série différenciée.

### 4.2 Entraînement et prévision

In [ ]:
from darts import TimeSeries
from darts.models import VARIMA

serie_train = TimeSeries.from_dataframe(train_desaison, freq='D')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    modele_varima = VARIMA(p=p, d=d, q=0).fit(serie_train)
    prevision_varima = modele_varima.predict(N_TEST)

# Retour à la consommation réelle : on réintègre la saisonnalité annuelle
prevision_desaison = prevision_varima.to_dataframe()['consommation'].to_numpy()
previsions['VARIMA (consommation + température)'] = pd.Series(
    prevision_desaison + saisonnalite_future('consommation', N_TEST), index=test.index
)

In [ ]:
# Référence : ARIMA sur la consommation seule, avec les mêmes ordres
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    arima = sm.tsa.ARIMA(train_desaison['consommation'], order=(p, d, 0)).fit()
previsions['ARIMA (consommation seule)'] = pd.Series(
    arima.forecast(N_TEST).to_numpy() + saisonnalite_future('consommation', N_TEST), index=test.index
)

for nom in ('VARIMA (consommation + température)', 'ARIMA (consommation seule)'):
    resultats[nom] = {
        'MAPE (%)': mean_absolute_percentage_error(y_test, previsions[nom]),
        'MAE (MW)': mean_absolute_error(y_test, previsions[nom]),
        'RMSE (MW)': np.sqrt(mean_squared_error(y_test, previsions[nom])),
    }

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_meteo.index[-(N_TEST + 365):], y.iloc[-(N_TEST + 365):], label='Consommation réelle', color='#7a7974', lw=1)
for nom, couleur in (('VARIMA (consommation + température)', '#1baf7a'), ('ARIMA (consommation seule)', '#2a78d6')):
    plt.plot(previsions[nom].index, previsions[nom], label=f"{nom} (MAPE : {resultats[nom]['MAPE (%)']:.2f} %)", color=couleur, lw=1.5)
plt.axvline(test.index[0], color='#7a7974', ls=':', lw=1)
plt.legend()
plt.ylabel('Consommation (MW)')
plt.title(f'Prévisions VARIMA({p},{d},0) et ARIMA({p},{d},0) sur les deux dernières années')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Comparaison

In [ ]:
pd.DataFrame(resultats).T.sort_values('MAPE (%)').round(2)

**Lecture des résultats**

- **XGBoost** : la comparaison avec et sans température mesure directement l'apport de la météo. Rappel : la température de la période de test est la température observée, donc cet apport est un maximum.
- **VARIMA** : sur un horizon de deux ans, les prévisions d'un modèle autorégressif convergent vite vers une valeur moyenne. Le VARIMA ne connaît pas la température future, qu'il prévoit lui-même : la température l'aide surtout à court terme. Sur un long horizon, l'essentiel de la forme de la prévision vient de la saisonnalité annuelle réintégrée.
- Pour exploiter vraiment la météo en prévision, un modèle qui prend la température comme **variable exogène connue** (ARIMAX / SARIMAX avec `exog`, ou XGBoost) est plus adapté qu'un VARIMA.